# Carbon Arc Data Library

This notebook demonstrates how to explore the complete Carbon Arc Data Library:

| Type | Count | Description |
|------|-------|-------------|
| **Attention** | 15 | App usage, streaming, web traffic, box office, advertising |
| **Wallet** | 12 | Credit cards, POS, receipts, transaction data |
| **Supply** | 10 | Job data, company relationships, workforce, firmographics |
| **Logistics** | 8 | Freight, weather, foot traffic, trade, transportation |
| **Balance Sheet** | 4 | Medical claims, vehicle registration, housing permits |
| **Reference** | 6 | Census, financial filings, macro data, regulatory |

## Setup

In [ ]:
import os
import json
from dotenv import load_dotenv
from carbonarc import CarbonArcClient
import pandas as pd

load_dotenv()

# Read in environment variables
API_AUTH_TOKEN = os.getenv("API_AUTH_TOKEN")

# Create API Client
ca = CarbonArcClient(API_AUTH_TOKEN)
print("Client initialized successfully!")

## 1. List All Datasets

Get the complete list of available datasets in the Carbon Arc Data Library.

In [ ]:
# Fetch all datasets
datasets = ca.data.get_datasets()

print(f"Total datasets: {datasets['size']}")
print(f"Total pages: {datasets['total_pages']}")

In [ ]:
# Create a summary DataFrame
summary_data = []
for ds in datasets['datasources']:
    data_info = ds.get('data', {})
    summary_data.append({
        'dataset_id': ds['dataset_id'][0],
        'name': ds['dataset_name'],
        'provider': ds.get('provider_name') or '',
        'type': data_info.get('Type', 'N/A'),
        'frequency': data_info.get('Frequency', 'N/A'),
        'history': data_info.get('History', 'N/A'),
        'lag': data_info.get('Lag', 'N/A'),
        'bulk_data': data_info.get('Bulk Data', 'N/A'),
        'description': ds.get('description', '')[:100] + '...' if len(ds.get('description', '')) > 100 else ds.get('description', '')
    })

df_datasets = pd.DataFrame(summary_data)
df_datasets.head(10)

## 2. Datasets by Type

Group datasets by their type to understand the distribution.

In [ ]:
# Count datasets by type
type_counts = df_datasets['type'].value_counts()
print("Datasets by Type:")
print("=" * 40)
for dtype, count in type_counts.items():
    print(f"{dtype:20} {count:3} datasets")

In [ ]:
# Filter datasets by type
def get_datasets_by_type(dtype):
    """Filter datasets by type."""
    return df_datasets[df_datasets['type'] == dtype][['dataset_id', 'name', 'provider', 'frequency']]

# Example: Get all Wallet datasets
print("Wallet Datasets:")
get_datasets_by_type('Wallet')

In [ ]:
# Example: Get all Attention datasets
print("Attention Datasets:")
get_datasets_by_type('Attention')

## 3. Dataset Details

Get detailed information about a specific dataset.

In [ ]:
# Get detailed information for a specific dataset
dataset_id = "CA0056"  # Credit Card - US Complete Panel
dataset_info = ca.data.get_dataset_information(dataset_id)

print(f"Dataset: {dataset_info['dataset_name']}")
print(f"ID: {dataset_id}")
print("\nDescription:")
print(dataset_info['description'])

In [ ]:
# Display all metadata for the dataset
data_fields = dataset_info.get('data', {})

print("\nDataset Metadata:")
print("=" * 60)
for key, value in data_fields.items():
    if isinstance(value, list):
        print(f"\n{key}:")
        for item in value[:5]:  # Show first 5 items
            print(f"  - {item}")
        if len(value) > 5:
            print(f"  ... and {len(value) - 5} more")
    elif isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

## 4. Find Datasets with Bulk Data

Identify datasets that offer bulk data exports.

In [ ]:
# Find datasets with bulk data available
bulk_datasets = df_datasets[df_datasets['bulk_data'] == 'Available']

print(f"Datasets with Bulk Data Available: {len(bulk_datasets)}")
print("=" * 60)
bulk_datasets[['dataset_id', 'name', 'type', 'frequency']]

## 5. Search Datasets by Keyword

In [ ]:
def search_datasets(keyword):
    """Search datasets by keyword in name or description."""
    keyword = keyword.lower()
    mask = (
        df_datasets['name'].str.lower().str.contains(keyword) |
        df_datasets['description'].str.lower().str.contains(keyword)
    )
    return df_datasets[mask][['dataset_id', 'name', 'type', 'description']]

# Example: Search for "credit card" datasets
print("Datasets related to 'credit card':")
search_datasets('credit card')

In [ ]:
# Search for healthcare-related datasets
print("Datasets related to 'health':")
search_datasets('health')

In [ ]:
# Search for music-related datasets
print("Datasets related to 'music':")
search_datasets('music')

## 6. Graph Data

The Data Library also includes graph datasets for network analysis.

In [ ]:
# Get available graphs
graphs = ca.data.get_graphs()

print(f"Available Graphs: {len(graphs.get('graphs', []))}")
for graph in graphs.get('graphs', []):
    print(f"\n{graph['graph_name']} ({graph['graph_id']})")
    print(f"  {graph.get('description', 'No description')[:100]}...")

## 7. Export Dataset Summary

In [ ]:
# Export full dataset summary to CSV
df_datasets.to_csv('data_library_summary.csv', index=False)
print("Exported to data_library_summary.csv")

# Display summary statistics
print(f"\nData Library Summary:")
print(f"=" * 40)
print(f"Total Datasets: {len(df_datasets)}")
print(f"Unique Providers: {df_datasets['provider'].replace('', pd.NA).dropna().nunique()}")
print(f"Bulk Data Available: {len(bulk_datasets)}")
print(f"\nBy Frequency:")
print(df_datasets['frequency'].value_counts().head(10))

## 8. Full Dataset Reference

Quick reference table of all datasets.

In [ ]:
# Display all datasets in a formatted table
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 50)

df_datasets[['dataset_id', 'name', 'type', 'provider', 'frequency', 'bulk_data']].sort_values('type')